# 04 Post-Generation Harmonization

This notebook is dedicated to **post-generation cleanup** after face-DragGAN notebook `05_diffusion_refinement.ipynb`.
It does **not** run a new diffusion pass. Instead, it takes the already generated result and performs a more
classical, controllable round of image harmonization that is easier to explain, tune, and compare.

## Why this notebook exists

After notebook `05`, the replacement dog is usually already plausible, but some artifacts can still remain:

- a pale or foggy halo around the subject boundary
- slight color or brightness mismatch between the pasted dog and the original scene
- local boundary transitions that still look like a pasted layer

These issues are often **post-generation consistency problems**, not failures of pose alignment or segmentation.
That is why this notebook focuses on **harmonization rather than regeneration**.

## Inputs

This notebook reads the outputs of notebook `05`, especially:

- the refined final image
- the pre-diffusion seed composite
- subject boundary masks
- the LAB-harmonized output produced here

## Outputs

For each pair, this notebook exports:

- the input image coming from notebook `05`
- the LAB-harmonized result
- the final postprocessed result
- zoomed crops and diagnostic masks for presentation

## Presentation goal

This notebook is intentionally structured so that someone familiar with computer vision can quickly understand:

1. what visual inconsistency remains after diffusion,
2. what local postprocessing operation is applied,
3. what each operation changes,
4. and why the final image looks more integrated into the original scene.


## Step 06 - Post-Generation Harmonization

This is the existing post-generation polish stage, connected to the true-DragGAN face branch diffusion output.

In [ ]:
import json
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import ipywidgets as widgets
import numpy as np
from IPython.display import display, clear_output
from PIL import Image


def find_project_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data" / "outputs" / "face_draggan_pipeline" / "05_diffusion_refine").exists() and (candidate / "notebooks").exists():
            return candidate
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "environment.yml").exists() or (candidate / ".git").exists():
            return candidate
    return cwd


def print_pair_header(title):
    print("\n" + "=" * 96)
    print(title)
    print("=" * 96)


PROJECT_ROOT = find_project_root()
REFINE_ROOT = PROJECT_ROOT / "data" / "outputs" / "face_draggan_pipeline" / "05_diffusion_refine"
POST_ROOT = PROJECT_ROOT / "data" / "outputs" / "face_draggan_pipeline" / "06_postprocess"
ROUTE_MODE = "eval_face_draggan_fast_pti_slides_pair_3_face_draggan_fast_pti_forward"

REFINE_ROUTE_ROOT = REFINE_ROOT / ROUTE_MODE if (REFINE_ROOT / ROUTE_MODE).exists() else REFINE_ROOT
OUTPUT_ROOT = POST_ROOT / ROUTE_MODE
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
MANIFEST_PATH = REFINE_ROUTE_ROOT / "batch_manifest.json"
if not MANIFEST_PATH.exists():
    raise FileNotFoundError(f"No notebook 05 manifest found at {MANIFEST_PATH}. Run face-DragGAN notebook 05 first for the selected route.")

with open(MANIFEST_PATH, "r", encoding="utf-8") as f:
    REFINE_JOBS = json.load(f)

print("Project root:", PROJECT_ROOT)
print("Route mode:", ROUTE_MODE)
print("Refine route root:", REFINE_ROUTE_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Manifest path:", MANIFEST_PATH)
print("Discovered jobs:", len(REFINE_JOBS))
for item in REFINE_JOBS:
    print(" -", item["pair_name"])

## Method Overview

This stage keeps the geometry and identity produced earlier, and only improves **scene compatibility**.

### Design principle

The goal is **not** to redraw the dog. The goal is to make the already generated dog feel less like a pasted
layer and more like part of the original photograph.

### Processing stages

1. **Boundary mask construction**

   We derive only the masks needed for LAB harmonization:

   - a subject-edge band inside the dog silhouette
   - a nearby background reference ring

   No edge polish, cloning, or shadow synthesis is applied.

2. **Boundary LAB harmonization**

   The dog boundary is adjusted in LAB color space using nearby background statistics.
   This is a local, conservative correction aimed at reducing visible color spill, haze, or brightness mismatch.

3. **Mild tone and sharpening**

   The final step applies a very small subject-only luminance/contrast adjustment and unsharp mask.

### Why postprocess after diffusion

Diffusion is useful for filling holes and repairing local seams, but it is not always the best tool for
small photometric mismatches. This notebook keeps only the conservative part that has been useful in practice:
local LAB color/luminance harmonization plus mild sharpening.


In [ ]:
def read_rgb(path):
    return np.array(Image.open(path).convert("RGB"))


def read_mask(path, threshold=127):
    return np.array(Image.open(path).convert("L")) > threshold


def ensure_uint8(image):
    return np.clip(image, 0, 255).astype(np.uint8)


def save_rgb(path, image):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray(ensure_uint8(image)).save(path)


def save_mask(path, mask):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    Image.fromarray((mask.astype(np.uint8) * 255)).save(path)


def plot_images(items, cols=3, figsize=(18, 12)):
    rows = int(np.ceil(len(items) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(rows, cols)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (title, image) in zip(axes.flat, items):
        if image.ndim == 2:
            ax.imshow(image, cmap="gray")
        else:
            ax.imshow(image)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    return fig


def overlay_mask(image, mask, color=(255, 80, 80), overlay_alpha=0.42):
    out = image.astype(np.float32).copy()
    color_arr = np.array(color, dtype=np.float32)
    mask3 = mask[..., None].astype(np.float32)
    out = out * (1.0 - overlay_alpha * mask3) + color_arr * (overlay_alpha * mask3)
    return ensure_uint8(out)


def soft_blend(original, generated, mask, blur_kernel=21):
    if blur_kernel % 2 == 0:
        blur_kernel += 1
    soft = cv2.GaussianBlur(mask.astype(np.float32), (blur_kernel, blur_kernel), 0)[..., None]
    return ensure_uint8(generated.astype(np.float32) * soft + original.astype(np.float32) * (1.0 - soft))


def mask_to_bbox(mask):
    ys, xs = np.where(mask > 0)
    if len(xs) == 0 or len(ys) == 0:
        return None
    return [int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1]


def crop_around_mask(image, mask, margin=64):
    bbox = mask_to_bbox(mask)
    if bbox is None:
        return image
    x1, y1, x2, y2 = bbox
    h, w = image.shape[:2]
    x1 = max(0, x1 - margin)
    y1 = max(0, y1 - margin)
    x2 = min(w, x2 + margin)
    y2 = min(h, y2 + margin)
    return image[y1:y2, x1:x2]

In [ ]:
POST_LAB_STRENGTH_L = 0.42
POST_LAB_STRENGTH_AB = 0.18
POST_TONE_STRENGTH = 0.10
POST_CONTRAST_GAIN = 1.025
POST_SHARPEN_AMOUNT = 0.16
POST_SHARPEN_SIGMA = 0.9
POST_SUBJECT_ALPHA_BLUR = 21


def build_boundary_masks(subject_mask, outer_width=25, inner_width=7, inner_band_width=9):
    subject_u8 = subject_mask.astype(np.uint8)
    outer = cv2.dilate(subject_u8, np.ones((outer_width, outer_width), np.uint8), iterations=1).astype(bool)
    outer_inner = cv2.dilate(subject_u8, np.ones((inner_width, inner_width), np.uint8), iterations=1).astype(bool)
    background_ring = np.logical_and(outer, ~outer_inner)
    core = cv2.erode(subject_u8, np.ones((inner_band_width, inner_band_width), np.uint8), iterations=1).astype(bool)
    subject_edge_band = np.logical_and(subject_mask, ~core)
    return subject_edge_band, background_ring


def boundary_lab_harmonize(image, subject_mask, reference_background, strength_L=POST_LAB_STRENGTH_L, strength_AB=POST_LAB_STRENGTH_AB, blur_kernel=19):
    edge_band, background_ring = build_boundary_masks(subject_mask)
    if edge_band.sum() < 20 or background_ring.sum() < 20:
        return image.copy(), edge_band, background_ring
    img_lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB).astype(np.float32)
    ref_lab = cv2.cvtColor(reference_background, cv2.COLOR_RGB2LAB).astype(np.float32)
    out_lab = img_lab.copy()
    strengths = [strength_L, strength_AB, strength_AB]
    for ch, strength in enumerate(strengths):
        subject_vals = img_lab[..., ch][edge_band]
        bg_vals = ref_lab[..., ch][background_ring]
        subj_mean = float(subject_vals.mean())
        subj_std = float(subject_vals.std() + 1e-4)
        bg_mean = float(bg_vals.mean())
        bg_std = float(bg_vals.std() + 1e-4)
        matched = ((subject_vals - subj_mean) / subj_std) * bg_std + bg_mean
        blended = (1.0 - strength) * subject_vals + strength * matched
        out_lab[..., ch][edge_band] = np.clip(blended, 0.0, 255.0)
    harmonized = cv2.cvtColor(np.clip(out_lab, 0.0, 255.0).astype(np.uint8), cv2.COLOR_LAB2RGB)
    edge_soft = cv2.GaussianBlur(edge_band.astype(np.float32), (blur_kernel, blur_kernel), 0) > 0.01
    return soft_blend(image, harmonized, edge_soft, blur_kernel=blur_kernel), edge_band, background_ring


def make_subject_soft_alpha(subject_mask, blur_kernel=POST_SUBJECT_ALPHA_BLUR):
    if blur_kernel % 2 == 0:
        blur_kernel += 1
    alpha = cv2.GaussianBlur(subject_mask.astype(np.float32), (blur_kernel, blur_kernel), 0)
    return np.clip(alpha, 0.0, 1.0)[..., None]


def mild_tone_and_sharpen(image, subject_mask, background_ring, tone_strength=POST_TONE_STRENGTH, contrast_gain=POST_CONTRAST_GAIN, sharpen_amount=POST_SHARPEN_AMOUNT, sharpen_sigma=POST_SHARPEN_SIGMA):
    if subject_mask.sum() < 20:
        return image.copy()
    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB).astype(np.float32)
    subject_vals = lab[..., 0][subject_mask]
    if background_ring.sum() >= 20:
        bg_vals = lab[..., 0][background_ring]
        brightness_delta = np.clip(float(bg_vals.mean() - subject_vals.mean()), -10.0, 10.0) * float(tone_strength)
    else:
        brightness_delta = 0.0
    subject_mean = float(subject_vals.mean())
    adjusted_l = (lab[..., 0][subject_mask] - subject_mean) * float(contrast_gain) + subject_mean + brightness_delta
    lab[..., 0][subject_mask] = np.clip(adjusted_l, 0.0, 255.0)
    toned = cv2.cvtColor(np.clip(lab, 0.0, 255.0).astype(np.uint8), cv2.COLOR_LAB2RGB)
    blurred = cv2.GaussianBlur(toned, (0, 0), float(sharpen_sigma))
    sharpened = cv2.addWeighted(toned, 1.0 + float(sharpen_amount), blurred, -float(sharpen_amount), 0)
    alpha = make_subject_soft_alpha(subject_mask)
    return ensure_uint8(sharpened.astype(np.float32) * alpha + image.astype(np.float32) * (1.0 - alpha))


def run_postprocess(refined_image, original_image, subject_mask):
    lab_result, edge_band, background_ring = boundary_lab_harmonize(refined_image, subject_mask, original_image)
    final_result = mild_tone_and_sharpen(lab_result, subject_mask, background_ring)
    return {
        "lab_result": lab_result,
        "final_result": final_result,
        "edge_band_mask": edge_band,
        "background_ring_mask": background_ring,
    }


## Processing Parameters and What They Mean

This notebook uses conservative LAB harmonization plus mild tone/sharpening. The goal is to improve boundary color,
luminance, and crispness without adding cloned texture or synthetic shadows.

### Key parameter groups

- **Boundary widths**
  These control how thick the inside and outside boundary regions are. Wider bands give the algorithm more room to
  harmonize, but can also make the effect too broad.

- **LAB harmonization strength**
  This controls how strongly the boundary colors are pulled toward the local background neighborhood. Increasing it can
  reduce halos, but too much can flatten fur detail.

- **Mild tone and sharpening**
  After LAB harmonization, the final image receives a small subject-only luminance/contrast adjustment and unsharp mask.
  This uses current-image statistics only, not texture copied from the original dog.

### How to read the visualizations

The notebook shows both full-image views and zoomed views.

- **Full-image panels** help judge whether the overall replacement remains natural in the scene.
- **Zoomed panels** are where you should inspect seam quality, halo suppression, edge color transitions.

The diagnostic masks are especially useful when presenting results to others:

- `subject_edge_band_mask` shows where boundary color harmonization is allowed.
- `background_ring_mask` shows which nearby background region is used as local context.

In short: if the full image still looks coherent and the zoomed crop loses the obvious pasted-layer look, then the
postprocessing stage is doing its job.


In [ ]:
POSTPROCESS_RESULTS = []

for pair_index, item in enumerate(REFINE_JOBS, start=1):
    run_config_path = Path(item["run_config_path"])
    with open(run_config_path, "r", encoding="utf-8") as f:
        refine_meta = json.load(f)
    warp_meta_path = Path(refine_meta["warp_metadata_path"])
    with open(warp_meta_path, "r", encoding="utf-8") as f:
        warp_meta = json.load(f)

    pair_name = item["pair_name"]
    output_dir = OUTPUT_ROOT / pair_name
    output_dir.mkdir(parents=True, exist_ok=True)

    refined_final = read_rgb(Path(refine_meta["outputs"]["refined_final"]))
    pre_diffusion_seed = read_rgb(Path(refine_meta["outputs"]["pre_diffusion_seed"]))
    original_image = read_rgb(Path(warp_meta["cutout_metadata_path"]).parent / "original_image.png")
    subject_mask = read_mask(Path(warp_meta["outputs"]["warped_mask"]))
    print_pair_header(f"[06] Pair {pair_index}/{len(REFINE_JOBS)} :: {pair_name}")
    print("Refine run config:", run_config_path)
    print("Warp metadata:", warp_meta_path)

    result = run_postprocess(refined_final, original_image, subject_mask)

    comparison_items = [
        ("Original image", original_image),
        ("Pre-diffusion seed", pre_diffusion_seed),
        ("Refined final from 03", refined_final),
        ("Boundary LAB harmonized", result["lab_result"]),
        ("Mild tone + sharpen final", result["final_result"]),
        ("Subject edge band", result["edge_band_mask"]),
        ("Background reference ring", result["background_ring_mask"]),
    ]
    plot_images(comparison_items, cols=3, figsize=(18, 16))

    zoom_items = [
        ("Zoom refined final", crop_around_mask(refined_final, subject_mask)),
        ("Zoom LAB harmonized", crop_around_mask(result["lab_result"], subject_mask)),
        ("Zoom mild tone + sharpen final", crop_around_mask(result["final_result"], subject_mask)),
    ]
    plot_images(zoom_items, cols=3, figsize=(18, 10))

    save_rgb(output_dir / "refined_input.png", refined_final)
    save_rgb(output_dir / "lab_harmonized.png", result["lab_result"])
    save_rgb(output_dir / "postprocessed_final.png", result["final_result"])
    save_rgb(output_dir / "zoom_postprocessed_final.png", crop_around_mask(result["final_result"], subject_mask))
    save_mask(output_dir / "subject_edge_band_mask.png", result["edge_band_mask"])
    save_mask(output_dir / "background_ring_mask.png", result["background_ring_mask"])

    run_summary = {
        "pair_name": pair_name,
        "route_mode": ROUTE_MODE,
        "source_refine_run_config": str(run_config_path),
        "source_warp_metadata": str(warp_meta_path),
        "outputs": {
            "refined_input": str(output_dir / "refined_input.png"),
            "lab_harmonized": str(output_dir / "lab_harmonized.png"),
            "postprocessed_final": str(output_dir / "postprocessed_final.png"),
            "zoom_postprocessed_final": str(output_dir / "zoom_postprocessed_final.png"),
            "subject_edge_band_mask": str(output_dir / "subject_edge_band_mask.png"),
            "background_ring_mask": str(output_dir / "background_ring_mask.png"),
        },
    }
    with open(output_dir / "run_summary.json", "w", encoding="utf-8") as f:
        json.dump(run_summary, f, ensure_ascii=False, indent=2)

    print("Saved post-processing package to:", output_dir)
    POSTPROCESS_RESULTS.append(
        {
            "pair_name": pair_name,
            "route_mode": ROUTE_MODE,
            "output_dir": str(output_dir),
            "run_summary_path": str(output_dir / "run_summary.json"),
        }
    )

with open(OUTPUT_ROOT / "batch_manifest.json", "w", encoding="utf-8") as f:
    json.dump(POSTPROCESS_RESULTS, f, ensure_ascii=False, indent=2)

print_pair_header("[06] Batch summary")
print("Saved manifest:", OUTPUT_ROOT / "batch_manifest.json")
for row in POSTPROCESS_RESULTS:
    print(" -", row["pair_name"], "->", row["output_dir"])

        ## How to Present the Results

        A good way to present this notebook is to walk through the outputs in the following order:

        1. **Start with the refined input from notebook 05**
           Explain that the replacement is already structurally plausible, but still shows minor integration artifacts.

        2. **Show the diagnostic masks**
           This makes it clear that notebook `06` is not changing the entire image. It is only acting on carefully
           selected boundary LAB regions.

        3. **Compare the intermediate outputs**

           - `lab_harmonized.png`: local color and luminance adjustment
           - `postprocessed_final.png`: the same LAB-harmonized result, saved as the final output

        4. **Use the zoomed crop as the main quality check**
           The zoomed view is where halos, fogginess, edge contamination, and floating artifacts are easiest to judge.

        ## Suggested narration

        A concise way to explain this notebook to others is:

        > "Notebook 03 uses diffusion to repair and refine the local replacement region.
        > Notebook 06 then treats the result as a nearly-correct composite and applies boundary-aware
        > harmonization so that color and luminance match the original scene more naturally."

        ## Saved artifacts

        Each pair directory contains both presentation-ready images and debugging assets. This makes the notebook useful
        both for qualitative demos and for technical diagnosis when a postprocessing choice does not behave as expected.
        


## Interactive Resolution Preview

This final module is meant for **presentation-time comparison**.
It lets you take any `04` postprocessed result and preview how it looks when compressed to a chosen display resolution.

### Why this is useful

In practice, results are often viewed in resized form:

- slides
- reports
- browser-based demos
- side-by-side comparison boards

A result that looks slightly layered at native resolution may look much more coherent after moderate downscaling.
This module helps you inspect that effect intentionally instead of guessing.

### What this module does

- select one processed pair from notebook `06`
- choose a target longest-side resolution
- generate compressed preview images in notebook
- save those previews under the corresponding `04` output folder

The generated previews are **display-oriented derivatives**. They do not overwrite the full-resolution outputs.


In [ ]:
def resize_longest_side(image, target_long_side, interpolation_down=cv2.INTER_AREA, interpolation_up=cv2.INTER_CUBIC):
    h, w = image.shape[:2]
    long_side = max(h, w)
    if long_side == target_long_side:
        return image.copy(), 1.0
    scale = target_long_side / float(long_side)
    new_w = max(1, int(round(w * scale)))
    new_h = max(1, int(round(h * scale)))
    interp = interpolation_down if scale < 1.0 else interpolation_up
    resized = cv2.resize(image, (new_w, new_h), interpolation=interp)
    return resized, scale


def build_resolution_preview_assets(run_summary_path, target_long_side):
    run_summary_path = Path(run_summary_path)
    with open(run_summary_path, "r", encoding="utf-8") as f:
        summary = json.load(f)

    outputs = summary["outputs"]
    refined_input = read_rgb(Path(outputs["refined_input"]))
    postprocessed_final = read_rgb(Path(outputs["postprocessed_final"]))
    zoom_postprocessed_final = read_rgb(Path(outputs["zoom_postprocessed_final"]))

    refined_preview, refined_scale = resize_longest_side(refined_input, target_long_side)
    final_preview, final_scale = resize_longest_side(postprocessed_final, target_long_side)
    zoom_target = max(256, min(target_long_side, max(zoom_postprocessed_final.shape[:2])))
    zoom_preview, zoom_scale = resize_longest_side(zoom_postprocessed_final, zoom_target)

    preview_dir = run_summary_path.parent / "resolution_previews"
    preview_dir.mkdir(parents=True, exist_ok=True)

    refined_path = preview_dir / f"refined_input_long{target_long_side}.png"
    final_path = preview_dir / f"postprocessed_final_long{target_long_side}.png"
    zoom_path = preview_dir / f"zoom_postprocessed_final_long{zoom_target}.png"

    save_rgb(refined_path, refined_preview)
    save_rgb(final_path, final_preview)
    save_rgb(zoom_path, zoom_preview)

    return {
        "pair_name": summary["pair_name"],
        "route_mode": summary["route_mode"],
        "target_long_side": int(target_long_side),
        "preview_dir": str(preview_dir),
        "refined_preview_path": str(refined_path),
        "final_preview_path": str(final_path),
        "zoom_preview_path": str(zoom_path),
        "refined_scale": float(refined_scale),
        "final_scale": float(final_scale),
        "zoom_scale": float(zoom_scale),
        "refined_preview": refined_preview,
        "final_preview": final_preview,
        "zoom_preview": zoom_preview,
    }


preview_options = [(row["pair_name"], row["run_summary_path"]) for row in POSTPROCESS_RESULTS]
preview_pair_dropdown = widgets.Dropdown(
    options=preview_options,
    description="Pair:",
    layout=widgets.Layout(width="650px"),
    style={"description_width": "100px"},
)
preview_resolution_dropdown = widgets.Dropdown(
    options=[384, 512, 640, 768, 960, 1024],
    value=640,
    description="Long side:",
    layout=widgets.Layout(width="250px"),
    style={"description_width": "100px"},
)
preview_button = widgets.Button(
    description="Generate preview",
    button_style="info",
    tooltip="Generate a resized display preview and save it under the 04 output folder.",
)
preview_output = widgets.Output()


def on_generate_preview(_):
    with preview_output:
        clear_output(wait=True)
        asset = build_resolution_preview_assets(
            preview_pair_dropdown.value,
            preview_resolution_dropdown.value,
        )
        print_pair_header(f"[06] Resolution preview :: {asset['pair_name']}")
        print("Route mode:", asset["route_mode"])
        print("Target longest side:", asset["target_long_side"])
        print("Saved preview directory:", asset["preview_dir"])
        print("Refined-input scale:", f"{asset['refined_scale']:.3f}")
        print("Final-result scale:", f"{asset['final_scale']:.3f}")
        print("Zoom scale:", f"{asset['zoom_scale']:.3f}")
        fig = plot_images(
            [
                (f"Refined input long={asset['target_long_side']}", asset["refined_preview"]),
                (f"Postprocessed final long={asset['target_long_side']}", asset["final_preview"]),
                ("Zoom postprocessed final", asset["zoom_preview"]),
            ],
            cols=3,
            figsize=(18, 6),
        )
        display(fig)
        plt.close(fig)


preview_button.on_click(on_generate_preview)
display(
    widgets.VBox(
        [
            widgets.HBox([preview_pair_dropdown, preview_resolution_dropdown, preview_button]),
            preview_output,
        ]
    )
)
